In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
db_path = Path("../db.sqlite3")
if not db_path.exists():
    db_path = Path("../db.sqlite3")

csv_path = Path("restaurant_order_items_joined.csv")

conn = sqlite3.connect(db_path)

query = """
SELECT
    oi.id AS order_item_id,
    oi.order_id,
    o.datetime AS order_datetime,
    oi.product_id,
    p.name AS product_name,
    CAST(p.price AS REAL) AS product_price,
    oi.quantity
FROM restaurant_orderitem AS oi
JOIN restaurant_order AS o
    ON oi.order_id = o.id
JOIN restaurant_product AS p
    ON oi.product_id = p.id
ORDER BY oi.id;
"""

df_sql = pd.read_sql_query(query, conn)
conn.close()

df_sql.to_csv(csv_path, index=False)

df = pd.read_csv(csv_path, parse_dates=["order_datetime"])

print(f"Rows in dataframe: {len(df)}")
print(f"CSV file created: {csv_path}")
display(df.head())

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top_10_quantity = (
    df.groupby("product_name", as_index=False)["quantity"]
    .sum()
    .sort_values("quantity", ascending=False)
    .head(10)
)

display(top_10_quantity)

plt.figure(figsize=(8, 8))
plt.pie(
    top_10_quantity["quantity"],
    labels=top_10_quantity["product_name"],
    autopct="%1.1f%%",
    startangle=90,
)
plt.title("Top 10 Most Popular Products by Quantity")
plt.tight_layout()
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["item_price"] = df["product_price"] * df["quantity"]

top_10_revenue = (
    df.groupby("product_name", as_index=False)["item_price"]
    .sum()
    .sort_values("item_price", ascending=False)
    .head(10)
)

display(top_10_revenue)

plt.figure(figsize=(8, 8))
plt.pie(
    top_10_revenue["item_price"],
    labels=top_10_revenue["product_name"],
    autopct="%1.1f%%",
    startangle=90,
)
plt.title("Top 10 Products by Revenue")
plt.tight_layout()
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df["order_hour"] = df["order_datetime"].dt.hour

income_by_hour = (
    df.groupby("order_hour")["item_price"]
    .sum()
    .reindex(range(24), fill_value=0)
    .reset_index()
)

display(income_by_hour)

plt.figure(figsize=(12, 6))
plt.bar(income_by_hour["order_hour"], income_by_hour["item_price"])
plt.xticks(range(24))
plt.xlabel("Order Hour")
plt.ylabel("Total Income")
plt.title("Total Restaurant Income by Order Hour")
plt.tight_layout()
plt.show()

best_hour = income_by_hour.loc[income_by_hour["item_price"].idxmax()]
print(
    f"Highest income is at {int(best_hour['order_hour'])}:00 "
    f"with total income {best_hour['item_price']:.2f}."
)

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

df["order_day_of_week"] = pd.Categorical(
    df["order_datetime"].dt.day_name(),
    categories=day_order,
    ordered=True,
)

income_by_day = (
    df.groupby("order_day_of_week", observed=False)["item_price"]
    .sum()
    .reindex(day_order)
    .reset_index()
)

display(income_by_day)

plt.figure(figsize=(10, 6))
plt.bar(income_by_day["order_day_of_week"], income_by_day["item_price"])
plt.xlabel("Order Day Of The Week")
plt.ylabel("Total Income")
plt.title("Total Restaurant Income by Day of Week")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

best_day = income_by_day.loc[income_by_day["item_price"].idxmax()]
print(
    f"Highest income is on {best_day['order_day_of_week']} "
    f"with total income {best_day['item_price']:.2f}."
)

# Additional Analytics
Extra checks that help understand the restaurant sales data:
- total orders and order items;
- total revenue;
- average order value;
- date range in the dataset;
- products with the highest average quantity per order item.

In [ ]:
summary = pd.DataFrame(
    {
        "metric": [
            "Total order items",
            "Total unique orders",
            "Total revenue",
            "Average order value",
            "Dataset start date",
            "Dataset end date",
        ],
        "value": [
            len(df),
            df["order_id"].nunique(),
            round(df["item_price"].sum(), 2),
            round(df.groupby("order_id")["item_price"].sum().mean(), 2),
            df["order_datetime"].min(),
            df["order_datetime"].max(),
        ],
    }
)

display(summary)

avg_quantity_by_product = (
    df.groupby("product_name", as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        order_item_count=("order_item_id", "count"),
        avg_quantity_per_order_item=("quantity", "mean"),
        total_revenue=("item_price", "sum"),
    )
    .sort_values(["avg_quantity_per_order_item", "total_quantity"], ascending=False)
    .head(10)
)

display(avg_quantity_by_product)